## Step 1 · Setup
Import the libraries, fix the random seed, and point to the two data files with relative paths, so the notebook runs top to bottom on any machine and gives the same numbers twice.

In [1]:
# Import pandas, numpy, Path and sklearn (sklearn only to print its version)
# Set one random seed used by every model in this notebook
# Point to the 2022 (train) and 2023 (test) files with relative paths
# Print library versions and confirm both files are found

import sys
import numpy as np
import pandas as pd
import sklearn
from pathlib import Path

RANDOM_STATE = 42
DATA_DIR = Path("data")
TRAIN_FILE = DATA_DIR / "sparcs_fl_2022.csv"
TEST_FILE = DATA_DIR / "sparcs_fl_2023.csv"

print("Python ", sys.version.split()[0])
print("pandas ", pd.__version__)
print("numpy  ", np.__version__)
print("sklearn", sklearn.__version__)
print("Train file found:", TRAIN_FILE.exists())
print("Test file found: ", TEST_FILE.exists())

Python  3.14.2
pandas  3.0.6
numpy   2.5.3
sklearn 1.9.1
Train file found: True
Test file found:  True


**Findings.** Both files are found through relative paths, so the notebook does not depend on my computer's folder layout. One seed (42) controls every model below, which is what makes the numbers repeat on a second run.

## Step 2 · Load the raw files
Load 2022 and 2023 exactly as downloaded, every column as text, with blanks kept as empty strings instead of being converted to NaN. Nothing is cleaned yet: Step 3 has to see the problems the file hides before anything fixes them.

In [2]:
# Read both CSVs with every column as text and blanks kept as "" (not NaN)
# Print each file's shape
# Confirm both years have the same columns in the same order
# Confirm each file holds one discharge year and one service area, and count hospitals

raw_2022 = pd.read_csv(TRAIN_FILE, dtype=str, keep_default_na=False)
raw_2023 = pd.read_csv(TEST_FILE, dtype=str, keep_default_na=False)

print("2022 shape:", raw_2022.shape)
print("2023 shape:", raw_2023.shape)
print("Same columns, same order:", list(raw_2022.columns) == list(raw_2023.columns))
print("Discharge Year:", raw_2022["Discharge Year"].unique().tolist(), raw_2023["Discharge Year"].unique().tolist())
print("Service area:  ", raw_2022["Hospital Service Area"].unique().tolist(), raw_2023["Hospital Service Area"].unique().tolist())
print("Hospitals:", raw_2022["Facility Name"].nunique(), "in 2022,", raw_2023["Facility Name"].nunique(), "in 2023")

2022 shape: (134596, 33)
2023 shape: (137867, 33)
Same columns, same order: True
Discharge Year: ['2022'] ['2023']
Service area:   ['Finger Lakes'] ['Finger Lakes']
Hospitals: 17 in 2022, 17 in 2023


**Findings.** The 2022 file has 134,596 discharges and the 2023 file has 137,867, each with the same 33 columns in the same order, so one cleaning routine can serve both years. Each file holds a single discharge year and only the Finger Lakes service area (17 hospitals), which confirms the download filter worked and that the time split will be a clean 2022-versus-2023 cut.

## Step 3 · Data check: what the file hides
Before cleaning anything, count every problem that a null check would miss: numbers stored as text, codes that mean "missing", categories spelled two ways, blanks, duplicates, categories that exist only in the test year, and IDs that are not unique. Each count goes into data.md and the verification log; the fixes happen in Step 4.

In [3]:
# For each year, count:
#   1. Length of Stay values that are not whole numbers
#   2. Total Charges and Total Costs stored as text with commas
#   3. Severity of Illness code 0 (a "missing" code), and which DRGs carry it
#   4. Blank 3-digit ZIPs and the "OOS" (out of state) code
#   5. Exact duplicate rows
#   6. Placeholder values: Gender "U", Type of Admission "Not Available"
# Across years, check:
#   7. MDC codes whose description is spelled differently in 2022 and 2023
#   8. CCSR diagnosis codes that appear in 2023 but never in 2022
#   9. Whether Operating Certificate Number, Facility Name and Permanent Facility Id are one-to-one

raw = {2022: raw_2022, 2023: raw_2023}

for year, df in raw.items():
    print(f"--- {year} ---")
    los_text = df.loc[~df["Length of Stay"].str.isdigit(), "Length of Stay"]
    print("1. Length of Stay not a whole number:", los_text.value_counts().to_dict())
    print("2. Total Charges with a comma:", df["Total Charges"].str.contains(",").sum(),
          "| Total Costs with a comma:", df["Total Costs"].str.contains(",").sum())
    sev0 = df[df["APR Severity of Illness Code"] == "0"]
    print("3. Severity code 0 rows:", len(sev0), "| their DRGs:", sev0["APR DRG Code"].value_counts().to_dict(),
          "| blank severity description:", (df["APR Severity of Illness Description"] == "").sum())
    zipc = df["Zip Code - 3 digits"]
    print("4. Blank 3-digit ZIP:", (zipc == "").sum(), "| 'OOS' ZIP:", (zipc == "OOS").sum())
    print("5. Exact duplicate rows:", df.duplicated().sum())
    print("6. Gender 'U':", (df["Gender"] == "U").sum(),
          "| Type of Admission 'Not Available':", (df["Type of Admission"] == "Not Available").sum())

mdc = (raw_2022[["APR MDC Code", "APR MDC Description"]].drop_duplicates()
       .merge(raw_2023[["APR MDC Code", "APR MDC Description"]].drop_duplicates(),
              on="APR MDC Code", suffixes=("_2022", "_2023")))
changed = mdc[mdc["APR MDC Description_2022"] != mdc["APR MDC Description_2023"]]
print("\n7. MDC codes whose description changed:", len(changed), "of", len(mdc))
print("   example:", changed.iloc[0, 1], "->", changed.iloc[0, 2])

new_ccsr = set(raw_2023["CCSR Diagnosis Code"]) - set(raw_2022["CCSR Diagnosis Code"])
print("8. CCSR codes in 2023 never seen in 2022:", len(new_ccsr),
      "| 2023 rows with one:", raw_2023["CCSR Diagnosis Code"].isin(new_ccsr).sum())

ids = pd.concat(raw.values())[["Operating Certificate Number", "Permanent Facility Id", "Facility Name"]].drop_duplicates()
shared = ids.groupby("Operating Certificate Number")["Facility Name"].nunique()
multi = ids.groupby("Facility Name")["Operating Certificate Number"].nunique()
print("9. Certificate numbers shared by 2+ hospitals:", shared[shared > 1].to_dict())
print("   Hospitals with 2+ certificate numbers:", multi[multi > 1].to_dict())
print("   Facility Name <-> Permanent Facility Id one-to-one:",
      ids.groupby("Facility Name")["Permanent Facility Id"].nunique().max() == 1
      and ids.groupby("Permanent Facility Id")["Facility Name"].nunique().max() == 1)

--- 2022 ---
1. Length of Stay not a whole number: {'120 +': 286}
2. Total Charges with a comma: 134360 | Total Costs with a comma: 131413
3. Severity code 0 rows: 123 | their DRGs: {'955': 120, '956': 3} | blank severity description: 123
4. Blank 3-digit ZIP: 1762 | 'OOS' ZIP: 2133
5. Exact duplicate rows: 70
6. Gender 'U': 2 | Type of Admission 'Not Available': 7
--- 2023 ---
1. Length of Stay not a whole number: {'120 +': 265}
2. Total Charges with a comma: 137703 | Total Costs with a comma: 135529
3. Severity code 0 rows: 68 | their DRGs: {'955': 65, '956': 3} | blank severity description: 68
4. Blank 3-digit ZIP: 1824 | 'OOS' ZIP: 2045
5. Exact duplicate rows: 51
6. Gender 'U': 4 | Type of Admission 'Not Available': 12

7. MDC codes whose description changed: 22 of 26
   example: PREGNANCY, CHILDBIRTH AND THE PUERPERIUM -> PREGNANCY, CHILDBIRTH & THE PUERPERIUM
8. CCSR codes in 2023 never seen in 2022: 20 | 2023 rows with one: 42
9. Certificate numbers shared by 2+ hospitals: {'07

**Findings.** The file hides at least four problems a null check cannot see: Length of Stay is text because 286 stays (2022) read "120 +", nearly every charge and cost is text with commas, severity code 0 marks 123 ungroupable stays (DRGs 955/956) rather than a real severity level, and 22 of 26 MDC descriptions are spelled differently in 2023 ("AND" became "&"), so joining on descriptions would silently split categories — codes are used instead. Two operating certificate numbers each cover two hospitals and St. Joseph's has two numbers, while Facility Name and Permanent Facility Id match one-to-one, so the hospital feature will use Facility Name; the 70 duplicate rows, blank ZIPs and 20 test-only diagnosis codes (42 rows) are handled in Step 4.

## Step 4 · Clean the data and define the target
Fix each problem Step 3 found, remove the stays the decision does not apply to, and build `extended_stay`: 1 if a stay ran longer than the 75th-percentile length of stay for its DRG and severity level. The cutoffs come from 2022 only and are applied unchanged to 2023, so nothing about the test year leaks into the target.

In [4]:
# Write one cleaning function used on both years:
#   - Length of Stay: read "120 +" as 120 and convert to a whole number
#   - Total Charges and Total Costs: strip commas and convert to numbers
#   - Blank 3-digit ZIP: recode as "Unknown" ("OOS" stays as its own category)
#   - Build a DRG x severity key from the codes, not the descriptions
# Keep the exact duplicate rows: with no patient ID, two identical rows are most likely two patients
# Remove newborn stays (MDC 15) and severity code 0 (ungroupable DRGs 955/956)
# On 2022 only: compute each DRG x severity group's size and 75th-percentile length of stay
# Keep groups with 30+ cases in 2022; drop rows in smaller groups from both years
# extended_stay = 1 if length of stay is greater than its group's 2022 cutoff
# Print the row count after each step and the base rate for each year

def clean(df):
    out = df.copy()
    out["los"] = out["Length of Stay"].replace("120 +", "120").astype(int)
    out["total_charges"] = out["Total Charges"].str.replace(",", "", regex=False).astype(float)
    out["total_costs"] = out["Total Costs"].str.replace(",", "", regex=False).astype(float)
    out["zip3"] = out["Zip Code - 3 digits"].replace("", "Unknown")
    out["drg_sev"] = out["APR DRG Code"] + "_" + out["APR Severity of Illness Code"]
    return out

counts = {}
clean_years = {}
for year, df in {2022: raw_2022, 2023: raw_2023}.items():
    c = clean(df)
    counts[year] = {"loaded": len(c)}
    c = c[c["APR MDC Code"] != "15"]
    counts[year]["after removing newborns (MDC 15)"] = len(c)
    c = c[c["APR Severity of Illness Code"] != "0"]
    counts[year]["after removing severity 0"] = len(c)
    clean_years[year] = c

cutoffs = (clean_years[2022].groupby("drg_sev")["los"]
           .agg(n="size", p75=lambda s: s.quantile(0.75)))
kept = cutoffs[cutoffs["n"] >= 30]
print("DRG x severity groups in 2022:", len(cutoffs), "| with 30+ cases:", len(kept))

for year in (2022, 2023):
    c = clean_years[year]
    c = c[c["drg_sev"].isin(kept.index)].copy()
    c["p75_los"] = c["drg_sev"].map(kept["p75"])
    c["extended_stay"] = (c["los"] > c["p75_los"]).astype(int)
    counts[year]["after removing groups under 30"] = len(c)
    counts[year]["extended stays"] = int(c["extended_stay"].sum())
    counts[year]["base rate"] = round(c["extended_stay"].mean(), 4)
    clean_years[year] = c

train = clean_years[2022]
test = clean_years[2023]

summary = pd.DataFrame(counts)
print(summary.map(lambda v: f"{v:.1%}" if v < 1 else f"{v:,.0f}"))
print("Any missing values left in the new columns:",
      int(train[["los", "total_charges", "total_costs", "p75_los", "extended_stay"]].isna().sum().sum()))

DRG x severity groups in 2022: 1145 | with 30+ cases: 626
                                     2022     2023
loaded                            134,596  137,867
after removing newborns (MDC 15)  121,823  125,672
after removing severity 0         121,700  125,604
after removing groups under 30    115,840  118,866
extended stays                     23,848   24,331
base rate                           20.6%    20.5%
Any missing values left in the new columns: 0


**Findings.** After cleaning and three exclusions, 115,840 stays remain for training (2022) and 118,866 for testing (2023); the groups-under-30 rule keeps 626 of 1,145 DRG x severity groups while dropping only about 5% of rows. The extended-stay rate is 20.6% in 2022 and 20.5% in 2023 — below 25% because many whole-day stays tie exactly at the cutoff — and the near-identical rates mean the 2023 test year looks like the year the model learns from.

## Step 5 · Features and the time split
Pick only the columns a case manager would know on the day of admission, split by year (2022 trains, 2023 tests), and confirm that no after-discharge column made it in. Then check what the model will see in 2023 that it never saw in 2022, and set up an encoder that can handle it.

In [5]:
# List the day-one features from the checkpoint proposal
# List every column that is NOT known at admission, and confirm none of them are features
# Train on 2022 and test on 2023: no random split, because the model will be used on future admissions
# For each feature, count its categories in 2022 and the 2023 categories it never saw in training
# Fit a one-hot encoder on 2022 only:
#   categories with fewer than 30 training rows are grouped into one "infrequent" column,
#   and a 2023 category never seen in training is sent to that same column instead of crashing

from sklearn.preprocessing import OneHotEncoder

FEATURES = ["Facility Name", "Type of Admission", "Emergency Department Indicator",
            "Age Group", "Gender", "Payment Typology 1", "zip3",
            "CCSR Diagnosis Code", "APR MDC Code"]
NOT_KNOWN_AT_ADMISSION = ["APR DRG Code", "APR Severity of Illness Code", "APR Risk of Mortality",
                          "CCSR Procedure Code", "Patient Disposition", "Total Charges", "Total Costs",
                          "Length of Stay", "los", "total_charges", "total_costs", "drg_sev", "p75_los"]
TARGET = "extended_stay"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Leakage columns in X:", [c for c in NOT_KNOWN_AT_ADMISSION if c in FEATURES])

check = pd.DataFrame({
    "categories in 2022": X_train.nunique(),
    "2023 categories not in 2022": [len(set(X_test[c]) - set(X_train[c])) for c in FEATURES],
    "2023 rows affected": [int((~X_test[c].isin(set(X_train[c]))).sum()) for c in FEATURES],
})
print(check.to_string())

encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False)
encoder.fit(X_train)
print("Encoded columns:", encoder.transform(X_train).shape[1],
      "| 2023 rows encode without error:", encoder.transform(X_test).shape[0] == len(X_test))

X_train: (115840, 9) | X_test: (118866, 9)
Leakage columns in X: []
                                categories in 2022  2023 categories not in 2022  2023 rows affected
Facility Name                                   17                            0                   0
Type of Admission                                6                            0                   0
Emergency Department Indicator                   2                            0                   0
Age Group                                        5                            0                   0
Gender                                           2                            1                   1
Payment Typology 1                               9                            0                   0
zip3                                            33                            3                   8
CCSR Diagnosis Code                            410                           28                  56
APR MDC Code                    

**Findings.** The model sees nine day-one features and none of the thirteen after-discharge columns; DRG, severity and risk of mortality are held back for the leakage check later. Only 65 of 118,866 test rows (0.05%) carry a category the model never saw in 2022, mostly 28 diagnosis codes, and the encoder sends those, plus any category with fewer than 30 training rows, to one "infrequent" column per feature, giving 324 encoded columns that work on both years.

## Step 6 · Baselines first
Before fitting any real model, score two baselines on the 2023 test year: a model that knows nothing (always predicts "not extended") and the better simple rule a case manager could use without a model (each diagnosis category's 2022 extended-stay rate). Every model after this must beat the rule, not just the dummy.

In [6]:
# Write one function that scores any model the same way:
#   accuracy, precision, recall, AUC, and the four confusion-matrix counts in plain words
# Baseline 1: DummyClassifier(strategy="most_frequent"), fit on 2022 only
# Baseline 2: the simple rule
#   score = each CCSR diagnosis category's extended-stay rate in 2022
#   a 2023 code never seen in 2022 gets the overall 2022 rate
#   flag the stay if its score is above the 2022 base rate
# Put both in one results table; later models get added to the same table

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

def evaluate(name, y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"model": name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred),
            "AUC": roc_auc_score(y_true, y_score),
            "flagged": int(tp + fp), "caught (TP)": int(tp), "missed (FN)": int(fn), "false alarms (FP)": int(fp)}

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_score = dummy.predict_proba(X_test)[:, 1]

base_rate_2022 = y_train.mean()
dx_rate = train.groupby("CCSR Diagnosis Code")[TARGET].mean()
rule_score = X_test["CCSR Diagnosis Code"].map(dx_rate).fillna(base_rate_2022)
rule_pred = (rule_score > base_rate_2022).astype(int)

results = pd.DataFrame([
    evaluate("Dummy (always 'not extended')", y_test, dummy_pred, dummy_score),
    evaluate("Rule: diagnosis category's 2022 rate", y_test, rule_pred, rule_score),
]).set_index("model")
print("2022 base rate used as the rule's cut-off:", round(base_rate_2022, 4))
print(results.round(3).to_string())

2022 base rate used as the rule's cut-off: 0.2059
                                      accuracy  precision  recall    AUC  flagged  caught (TP)  missed (FN)  false alarms (FP)
model                                                                                                                         
Dummy (always 'not extended')            0.795       0.00   0.000  0.500        0            0        24331                  0
Rule: diagnosis category's 2022 rate     0.459       0.23   0.701  0.571    74129        17051         7280              57078


**Findings.** The dummy scores 79.5% accuracy and catches none of the 24,331 extended stays, which is why accuracy is the wrong yardstick here. The diagnosis-rate rule catches 70% of them but flags 74,129 of 118,866 admissions (62%) to do it, with only 23% of flags correct and an AUC of 0.571; that is the bar a real model must clear, and a case management team could not act on a list that long.

## Step 7 · Logistic regression
Fit the simplest real model first: one-hot encoding plus logistic regression in a single pipeline, so the encoder is learned from 2022 only. Check it with 5-fold cross-validation inside 2022 before touching 2023, then score it on the test year at the default 0.5 threshold. The threshold gets chosen properly, by cost, in a later step.

In [7]:
# Build a pipeline: one-hot encoder (same settings as Step 5) -> logistic regression
# Cross-validate AUC on 2022 only (5 folds, stratified, fixed seed) to see how stable it is
# Fit on all of 2022, then predict probabilities for 2023
# Compare train AUC to test AUC to check for overfitting
# Add it to the results table at the default 0.5 threshold
# Print the range of predicted probabilities

import time
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

logit = make_pipeline(
    OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
)
t0 = time.time()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(logit, X_train, y_train, cv=cv, scoring="roc_auc")
print("5-fold CV AUC on 2022:", cv_auc.round(3), "| mean", round(cv_auc.mean(), 3))

logit.fit(X_train, y_train)
logit_score = logit.predict_proba(X_test)[:, 1]
logit_pred = (logit_score >= 0.5).astype(int)
train_auc = roc_auc_score(y_train, logit.predict_proba(X_train)[:, 1])
print("Train AUC (2022):", round(train_auc, 3), "| Test AUC (2023):", round(roc_auc_score(y_test, logit_score), 3))
print(f"Fit time: {time.time()-t0:.0f} s")

row = evaluate("Logistic regression (threshold 0.5)", y_test, logit_pred, logit_score)
results.loc[row.pop("model")] = row
print(results.round(3).to_string())
print("Predicted probability, 2023: min", round(logit_score.min(), 3),
      "| median", round(np.median(logit_score), 3), "| max", round(logit_score.max(), 3))

5-fold CV AUC on 2022: [0.622 0.629 0.636 0.629 0.623] | mean 0.628
Train AUC (2022): 0.637 | Test AUC (2023): 0.628
Fit time: 4 s
                                      accuracy  precision  recall    AUC  flagged  caught (TP)  missed (FN)  false alarms (FP)
model                                                                                                                         
Dummy (always 'not extended')            0.795      0.000   0.000  0.500        0            0        24331                  0
Rule: diagnosis category's 2022 rate     0.459      0.230   0.701  0.571    74129        17051         7280              57078
Logistic regression (threshold 0.5)      0.795      0.408   0.004  0.628      223           91        24240                132
Predicted probability, 2023: min 0.018 | median 0.208 | max 0.679


**Findings.** Logistic regression ranks stays better than the diagnosis rule (test AUC 0.628 vs 0.571), and train AUC (0.637), cross-validated AUC (0.628) and test AUC (0.628) are nearly equal, so it is not overfitting and 2023 behaves like 2022. At the default 0.5 threshold it is useless — it flags only 223 stays and catches 91 — because almost no admission gets a probability above 0.5 (median 0.208, max 0.679), which is why the threshold has to be chosen from the cost of each mistake, not left at 0.5.

## Step 8 · Random forest
Fit a random forest to see whether a non-linear model ranks stays better than logistic regression. Its one setting that matters here, min_samples_leaf (the smallest group a leaf may hold), is chosen by 3-fold cross-validation on 2022 only, so the test year is still untouched when the choice is made.

In [8]:
# Build a pipeline: same one-hot encoder -> random forest (200 trees, fixed seed, all CPU cores)
# Try min_samples_leaf = 3, 10, 30, 100 with 3-fold cross-validation on 2022, scored by AUC
# Keep the setting with the best cross-validated AUC, refit on all of 2022
# Compare train AUC to test AUC to check for overfitting
# Add it to the results table at the default 0.5 threshold

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

forest = make_pipeline(
    OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30),
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
)
t0 = time.time()
search = GridSearchCV(forest, {"randomforestclassifier__min_samples_leaf": [3, 10, 30, 100]},
                      cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
                      scoring="roc_auc")
search.fit(X_train, y_train)
print("CV AUC by min_samples_leaf:",
      {int(k): round(float(v), 3) for k, v in zip(search.cv_results_["param_randomforestclassifier__min_samples_leaf"],
                                                  search.cv_results_["mean_test_score"])})
print("Chosen min_samples_leaf:", search.best_params_["randomforestclassifier__min_samples_leaf"])

forest = search.best_estimator_
forest_score = forest.predict_proba(X_test)[:, 1]
forest_pred = (forest_score >= 0.5).astype(int)
print("Train AUC (2022):", round(roc_auc_score(y_train, forest.predict_proba(X_train)[:, 1]), 3),
      "| Test AUC (2023):", round(roc_auc_score(y_test, forest_score), 3))
print(f"Search + fit time: {time.time()-t0:.0f} s")

row = evaluate("Random forest (threshold 0.5)", y_test, forest_pred, forest_score)
results.loc[row.pop("model")] = row
print(results.round(3).to_string())

CV AUC by min_samples_leaf: {3: 0.641, 10: 0.643, 30: 0.637, 100: 0.626}
Chosen min_samples_leaf: 10
Train AUC (2022): 0.692 | Test AUC (2023): 0.639
Search + fit time: 32 s
                                      accuracy  precision  recall    AUC  flagged  caught (TP)  missed (FN)  false alarms (FP)
model                                                                                                                         
Dummy (always 'not extended')            0.795      0.000   0.000  0.500        0            0        24331                  0
Rule: diagnosis category's 2022 rate     0.459      0.230   0.701  0.571    74129        17051         7280              57078
Logistic regression (threshold 0.5)      0.795      0.408   0.004  0.628      223           91        24240                132
Random forest (threshold 0.5)            0.795      0.667   0.000  0.639       12            8        24323                  4


**Findings.** Cross-validation on 2022 picked min_samples_leaf = 10 (CV AUC 0.643; smaller leaves overfit, larger ones underfit), and the forest edges out logistic regression on the test year, 0.639 vs 0.628 AUC, an improvement that also shows up in cross-validation (0.643 vs 0.628), so it is not a fluke of 2023. The gain is small and comes with a wider train-test gap (0.692 vs 0.639, against 0.637 vs 0.628 for logistic regression); like logistic regression, it flags almost nothing at 0.5, so which model to deploy is decided in the cost step, in dollars, not by AUC alone.

## Step 9 · What a mistake costs, and the threshold trade-off
Put a dollar value on each kind of mistake, then show what every threshold would do — how many admissions get flagged, how many extended stays get caught, and the net savings — using out-of-fold predictions on 2022 so the test year is still untouched. The threshold is chosen from this table in Step 10.

In [9]:
# Cost inputs, each with its source:
#   cost per inpatient day = median of Total Costs / Length of Stay, 2022 training rows (SPARCS)
#   days saved by early discharge planning = 0.73 (Cochrane review, via AAFP 2022)
#   early review = 0.5 hours of a registered nurse case manager (assumption)
#     at $45.42/hour (BLS national mean wage for registered nurses, May 2023)
# Net savings at a threshold = (extended stays caught x saving per catch) - (stays flagged x review cost)
# Get out-of-fold probabilities on 2022 for both models (5-fold CV), so no 2023 data is used
# For a range of thresholds, show: share flagged, flags per hospital per day, recall, precision, net savings

from sklearn.model_selection import cross_val_predict

COST_PER_DAY = (train["total_costs"] / train["los"]).median()
DAYS_SAVED = 0.73
SAVING_PER_CATCH = COST_PER_DAY * DAYS_SAVED
RN_HOURLY_WAGE = 45.42
REVIEW_HOURS = 0.5
REVIEW_COST = RN_HOURLY_WAGE * REVIEW_HOURS
print(f"Cost per inpatient day (2022 median): ${COST_PER_DAY:,.0f}")
print(f"Saving per extended stay caught:      ${SAVING_PER_CATCH:,.0f}  ({DAYS_SAVED} days x cost per day)")
print(f"Cost of one early review:             ${REVIEW_COST:,.2f}  ({REVIEW_HOURS} h x ${RN_HOURLY_WAGE}/h)")
print(f"A miss costs {SAVING_PER_CATCH / REVIEW_COST:.0f}x a wasted review | break-even probability {REVIEW_COST / SAVING_PER_CATCH:.3f}")

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = {"Logistic regression": cross_val_predict(logit, X_train, y_train, cv=cv5, method="predict_proba")[:, 1],
       "Random forest": cross_val_predict(forest, X_train, y_train, cv=cv5, method="predict_proba")[:, 1]}

y_arr = y_train.to_numpy()
n_hospitals = train["Facility Name"].nunique()
rows = []
for name, p in oof.items():
    for t in [0.014, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
        flag = p >= t
        tp, flagged = int((flag & (y_arr == 1)).sum()), int(flag.sum())
        rows.append({"model": name, "threshold": t,
                     "% admissions flagged": 100 * flagged / len(p),
                     "flags per hospital per day": flagged / 365 / n_hospitals,
                     "recall": tp / y_arr.sum(), "precision": tp / flagged if flagged else 0,
                     "net savings ($M)": (tp * SAVING_PER_CATCH - flagged * REVIEW_COST) / 1e6})
tradeoff = pd.DataFrame(rows)
print(tradeoff.round(3).to_string(index=False))

Cost per inpatient day (2022 median): $2,272
Saving per extended stay caught:      $1,659  (0.73 days x cost per day)
Cost of one early review:             $22.71  (0.5 h x $45.42/h)
A miss costs 73x a wasted review | break-even probability 0.014
              model  threshold  % admissions flagged  flags per hospital per day  recall  precision  net savings ($M)
Logistic regression      0.014               100.000                      18.669   1.000      0.206            36.924
Logistic regression      0.100                91.373                      17.058   0.966      0.218            35.804
Logistic regression      0.150                74.727                      13.951   0.862      0.237            32.114
Logistic regression      0.200                52.057                       9.718   0.668      0.264            25.047
Logistic regression      0.250                29.058                       5.425   0.414      0.293            15.614
Logistic regression      0.300               

**Findings.** At face value a missed extended stay costs 73x a wasted review ($1,659 vs $22.71), which puts the money-maximizing threshold at 0.014 and means reviewing every admission — a rule no case management team can staff, and one that makes any model pointless. I chose instead to treat review capacity as the binding constraint: the real cost of a false alarm is the extended-stay patient who missed a limited early-review slot, so the threshold is set to fill a capacity of 30% of admissions, where the random forest catches 44% of extended stays at 1.5x the base rate on 2022 out-of-fold predictions. The alternative (Option A) would have flagged 100% of admissions; a stricter 2x-lift bar would only be met by flagging the top 1%, catching 2% of extended stays.

## Step 10 · Lock the threshold, then test once on 2023
Turn the 30% capacity decision into a probability threshold using 2022 out-of-fold predictions only, then apply it, unchanged, to the 2023 test year in a single pass. The diagnosis rule and logistic regression get thresholds set the same way, so all three are compared at about the same review volume, in extended stays caught and dollars.

In [10]:
# Review capacity: 30% of admissions (my choice in Step 9)
# Threshold for each model = the 2022 score that only 30% of 2022 admissions reach
#   (out-of-fold for the two models; the in-sample diagnosis rate for the rule)
# Apply each threshold unchanged to 2023, one pass, no adjusting afterward
# Report the confusion matrix in words, recall, precision, lift over the base rate, and net savings
# State how much the chosen model beats the rule by, in stays caught and dollars

CAPACITY_SHARE = 0.30

THRESHOLD = float(np.quantile(oof["Random forest"], 1 - CAPACITY_SHARE))
logit_threshold = float(np.quantile(oof["Logistic regression"], 1 - CAPACITY_SHARE))
rule_train_score = X_train["CCSR Diagnosis Code"].map(dx_rate)
rule_threshold = float(np.quantile(rule_train_score, 1 - CAPACITY_SHARE))
print(f"Thresholds set on 2022 for a {CAPACITY_SHARE:.0%} review capacity:")
print(f"  random forest {THRESHOLD:.3f} | logistic regression {logit_threshold:.3f} | diagnosis rule {rule_threshold:.3f}")

def at_capacity(name, score, threshold):
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {"model": name, "% flagged": 100 * (tp + fp) / len(pred),
            "caught (TP)": int(tp), "false alarms (FP)": int(fp), "missed (FN)": int(fn), "correctly left (TN)": int(tn),
            "recall": tp / (tp + fn), "precision": tp / (tp + fp), "lift": (tp / (tp + fp)) / y_test.mean(),
            "net savings ($M)": (tp * SAVING_PER_CATCH - (tp + fp) * REVIEW_COST) / 1e6}

final = pd.DataFrame([
    at_capacity("Diagnosis rule", rule_score, rule_threshold),
    at_capacity("Logistic regression", logit_score, logit_threshold),
    at_capacity("Random forest (chosen)", forest_score, THRESHOLD),
]).set_index("model")
print("\n2023 test year, one pass:")
print(final.round(3).to_string())

gain = final.loc["Random forest (chosen)", "net savings ($M)"] - final.loc["Diagnosis rule", "net savings ($M)"]
print(f"\nRandom forest vs diagnosis rule: "
      f"{final.loc['Random forest (chosen)', 'caught (TP)'] - final.loc['Diagnosis rule', 'caught (TP)']:+,} extended stays caught, "
      f"${gain:+.2f}M net savings")

Thresholds set on 2022 for a 30% review capacity:
  random forest 0.244 | logistic regression 0.248 | diagnosis rule 0.227

2023 test year, one pass:
                        % flagged  caught (TP)  false alarms (FP)  missed (FN)  correctly left (TN)  recall  precision   lift  net savings ($M)
model                                                                                                                                          
Diagnosis rule             34.996        10032              31566        14299                62969   0.412      0.241  1.178            15.694
Logistic regression        30.952        10723              26068        13608                68467   0.441      0.291  1.424            16.950
Random forest (chosen)     30.894        10843              25880        13488                68655   0.446      0.295  1.442            17.150

Random forest vs diagnosis rule: +811 extended stays caught, $+1.46M net savings


**Findings.** With the threshold fixed at 0.244 from 2022, the random forest flags 30.9% of 2023 admissions and catches 10,843 of 24,331 extended stays (44.6%), with 29.5% of flags correct (1.44x the base rate) — close to its 2022 out-of-fold numbers, so the threshold carried over to a new year. Against the diagnosis rule it catches 811 more extended stays while flagging fewer admissions (30.9% vs 35.0%; the rule overshoots because many diagnoses share the same rate), worth an estimated $1.46M more in net savings; its edge over logistic regression is small (120 more stays caught, $0.20M), and all dollar figures rest on the 0.73-day saving applied to every caught stay.

## Step 11 · Leakage check and duplicate check
Two checks the verification log needs. First, refit the forest with the three columns coders assign after discharge (APR-DRG, severity of illness, risk of mortality) to measure how much the deployed model gives up by leaving them out. Second, refit without the 4 exact-duplicate training rows to confirm they do not move the result.

In [11]:
# Leakage check: same forest settings, add the three after-discharge columns, refit on 2022, score 2023
# Compare test AUC with and without them
# Duplicate check: drop the 4 exact-duplicate training rows, refit the same forest
# Compare test AUC, and recall at the locked threshold, with and without them

from sklearn.base import clone

LEAKY = ["APR DRG Code", "APR Severity of Illness Code", "APR Risk of Mortality"]
leaky_forest = clone(forest)
leaky_forest.fit(train[FEATURES + LEAKY], y_train)
leaky_score = leaky_forest.predict_proba(test[FEATURES + LEAKY])[:, 1]
print(f"Test AUC, day-one features only (deployed):      {roc_auc_score(y_test, forest_score):.3f}")
print(f"Test AUC, plus DRG, severity, risk of mortality: {roc_auc_score(y_test, leaky_score):.3f}")

raw_cols = list(raw_2022.columns)
keep_rows = ~train.duplicated(subset=raw_cols)
dedup_forest = clone(forest)
dedup_forest.fit(X_train[keep_rows], y_train[keep_rows])
dedup_score = dedup_forest.predict_proba(X_test)[:, 1]
print(f"\nDuplicate check: dropped {int((~keep_rows).sum())} training rows")
print(f"  test AUC: with duplicates {roc_auc_score(y_test, forest_score):.4f} | without {roc_auc_score(y_test, dedup_score):.4f}")
print(f"  recall at threshold {THRESHOLD:.3f}: with {recall_score(y_test, forest_score >= THRESHOLD):.4f} | "
      f"without {recall_score(y_test, dedup_score >= THRESHOLD):.4f}")

Test AUC, day-one features only (deployed):      0.639
Test AUC, plus DRG, severity, risk of mortality: 0.657

Duplicate check: dropped 4 training rows
  test AUC: with duplicates 0.6387 | without 0.6391
  recall at threshold 0.244: with 0.4456 | without 0.4503


**Findings.** Adding the three after-discharge columns lifts test AUC only from 0.639 to 0.657, because the target is already defined relative to each stay's DRG and severity group; the deployed model leaves them out because none exist at admission, and the 0.018 gap is the measured price of that honesty. Dropping the 4 duplicate training rows moves test AUC by 0.0004 and recall by under half a point — about what refitting a forest on slightly different rows does by chance — so keeping the duplicates does not change the result.

## Step 12 · What drives the predictions
A random forest has no coefficients, so explain it with permutation importance: shuffle one feature at a time on 2023 data and measure how much test AUC drops. Importance says how much the model leans on a feature, not which way it pushes, so the two strongest features also get their 2022 extended-stay rates by category.

In [12]:
# Permutation importance on a fixed random sample of 20,000 test rows (to keep run time short)
#   shuffle each of the 9 features 5 times, record the average AUC drop and its spread
# The forest's built-in importance is not used: it favors features with many categories
# For the top two features, show the 2022 extended-stay rate by category, to give direction

from sklearn.inspection import permutation_importance

sample = X_test.sample(n=20000, random_state=RANDOM_STATE)
imp = permutation_importance(forest, sample, y_test.loc[sample.index], scoring="roc_auc",
                             n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
importance = (pd.DataFrame({"AUC drop when shuffled": imp.importances_mean, "spread (sd)": imp.importances_std},
                           index=FEATURES)
              .sort_values("AUC drop when shuffled", ascending=False))
print(importance.round(4).to_string())

for col in ["Facility Name", "Age Group"]:
    direction = (train.groupby(col)[TARGET].agg(stays="size", extended_rate="mean")
                 .sort_values("extended_rate", ascending=False))
    print(f"\n2022 extended-stay rate by {col}:")
    print(direction.round(3).to_string())

                                AUC drop when shuffled  spread (sd)
Facility Name                                   0.0341       0.0016
Age Group                                       0.0298       0.0034
Type of Admission                               0.0240       0.0026
CCSR Diagnosis Code                             0.0195       0.0006
Payment Typology 1                              0.0137       0.0020
APR MDC Code                                    0.0106       0.0010
Emergency Department Indicator                  0.0085       0.0015
zip3                                            0.0074       0.0009
Gender                                          0.0022       0.0007

2022 extended-stay rate by Facility Name:
                                                        stays  extended_rate
Facility Name                                                               
Unity Specialty Hospital                                  320          0.466
Soldiers and Sailors Memorial Hospital of Yate

**Findings.** The forest leans most on which hospital the patient is in (shuffling it costs 0.034 AUC), then age group (0.030) and type of admission (0.024); gender barely matters (0.002). Patients 70 and older run long 24.9% of the time against 13.4% for children, and hospital rates range from 9.5% to 46.6% even though the target already adjusts for DRG and severity — a difference that likely reflects case mix within groups, post-acute bed availability and discharge practices, which this data cannot separate, so it describes where long stays happen, not why.

## Step 13 · Where the model is weakest
An overall AUC can hide groups where the model barely ranks better than chance. Score the 2023 test year separately for each segment (age group, admission type, medical vs surgical, payer, ED or not, body system), keep segments with at least 1,000 stays, and report the weakest. That segment goes in the model card and the app's limits panel.

In [13]:
# Attach the forest's 2023 score and its flag at the locked threshold to the test rows
# For each segment with 1,000+ stays and both outcomes present, compute:
#   number of stays, base rate, AUC, % flagged, and recall (share of its extended stays caught)
# Sort by AUC; print the five weakest, the overall AUC for reference, and the three strongest

seg_rows = []
scored = test.assign(score=forest_score, flagged=(forest_score >= THRESHOLD).astype(int))
for col in ["Age Group", "Type of Admission", "APR Medical Surgical Description",
            "Payment Typology 1", "Emergency Department Indicator", "APR MDC Description"]:
    for value, g in scored.groupby(col):
        if len(g) < 1000 or g[TARGET].nunique() < 2:
            continue
        seg_rows.append({"segment": f"{col} = {value}", "stays": len(g), "base rate": g[TARGET].mean(),
                         "AUC": roc_auc_score(g[TARGET], g["score"]),
                         "% flagged": g["flagged"].mean() * 100,
                         "recall": g.loc[g[TARGET] == 1, "flagged"].mean()})
segments = pd.DataFrame(seg_rows).sort_values("AUC")
print("Weakest five:")
print(segments.head(5).round(3).to_string(index=False))
print(f"\nOverall test AUC: {roc_auc_score(y_test, forest_score):.3f}")
print("Strongest three:")
print(segments.tail(3).round(3).to_string(index=False))

Weakest five:
                                                                             segment  stays  base rate   AUC  % flagged  recall
                                                       Payment Typology 1 = Medicare  62512      0.235 0.600     48.373   0.592
                                                             Age Group = 70 or Older  45456      0.241 0.600     55.317   0.665
            APR MDC Description = DISEASES & DISORDERS OF THE KIDNEY & URINARY TRACT   6358      0.213 0.600     38.990   0.498
APR MDC Description = INFECTIOUS & PARASITIC DISEASES, SYSTEMIC OR UNSPECIFIED SITES  11751      0.220 0.600     36.797   0.477
                                   APR MDC Description = MENTAL DISEASES & DISORDERS   4599      0.242 0.602     27.484   0.368

Overall test AUC: 0.639
Strongest three:
                                                     segment  stays  base rate   AUC  % flagged  recall
                                        Age Group = 18 to 29  10369     

**Findings.** The model is weakest for patients 70 and older (AUC 0.600 against 0.639 overall, 45,456 test stays): it flags 55% of them largely because they are old, but within that group it ranks barely better than chance, and Medicare patients (mostly the same people) show the same 0.600. Mental health admissions are the costliest miss — they run long 24.2% of the time but only 36.8% of those are caught — while the model ranks best for pregnancy-related stays (AUC 0.728), which a case management program needs least.

## Step 14 · Save the model and what the app needs
Save the chosen random forest, the locked threshold and cost inputs, the dropdown options the app will offer, and the scored 2023 test rows. The app loads these files instead of retraining, so it starts fast and shows exactly the numbers in this notebook. Reload the saved model to confirm it reproduces the same scores.

In [14]:
# Save the fitted forest pipeline (encoder + model) to model.joblib, compressed
# Save app_config.json: threshold, capacity, base rates, cost inputs, feature list,
#   dropdown options from 2022, CCSR code -> description, CCSR -> its most common MDC, MDC -> description,
#   and the scikit-learn version (a saved model must be loaded with the same version)
# Save the scored 2023 test rows (segment columns, outcome, score) to data/test_scores.csv
#   for the app's "how good is this" view and threshold slider
# Reload the model and confirm it gives identical 2023 scores; print file sizes

import json
import joblib

joblib.dump(forest, "model.joblib", compress=3)

ccsr_desc = train.groupby("CCSR Diagnosis Code")["CCSR Diagnosis Description"].first()
ccsr_to_mdc = train.groupby("CCSR Diagnosis Code")["APR MDC Code"].agg(lambda s: s.mode().iloc[0])
mdc_desc = test.groupby("APR MDC Code")["APR MDC Description"].first()

config = {
    "threshold": THRESHOLD,
    "capacity_share": CAPACITY_SHARE,
    "train_year": 2022, "test_year": 2023,
    "base_rate_train": float(y_train.mean()), "base_rate_test": float(y_test.mean()),
    "cost_per_day": float(COST_PER_DAY), "days_saved": DAYS_SAVED,
    "rn_hourly_wage": RN_HOURLY_WAGE, "review_hours": REVIEW_HOURS,
    "features": FEATURES,
    "options": {c: sorted(X_train[c].unique().tolist()) for c in FEATURES},
    "ccsr_description": ccsr_desc.to_dict(),
    "ccsr_to_mdc": ccsr_to_mdc.to_dict(),
    "mdc_description": mdc_desc.to_dict(),
    "sklearn_version": sklearn.__version__,
}
with open("app_config.json", "w") as f:
    json.dump(config, f, indent=2)

test_scores = test[["Facility Name", "Age Group", "Type of Admission", "Payment Typology 1",
                                        "Emergency Department Indicator", "APR MDC Description", TARGET]].assign(score=forest_score, rule_score=rule_score)
test_scores.to_csv(DATA_DIR / "test_scores.csv", index=False)

reloaded = joblib.load("model.joblib")
same = np.allclose(reloaded.predict_proba(X_test)[:, 1], forest_score)
print("Saved model.joblib:        ", round(Path("model.joblib").stat().st_size / 1e6, 1), "MB")
print("Saved app_config.json:     ", round(Path("app_config.json").stat().st_size / 1e3, 1), "KB")
print("Saved data/test_scores.csv:", round((DATA_DIR / "test_scores.csv").stat().st_size / 1e6, 1), "MB,", len(test_scores), "rows")
print("Reloaded model gives identical 2023 scores:", same)
print("Threshold saved:", round(config["threshold"], 3), "| diagnosis codes in dropdown:", len(config["options"]["CCSR Diagnosis Code"]))

Saved model.joblib:         10.4 MB
Saved app_config.json:      42.5 KB
Saved data/test_scores.csv: 18.2 MB, 118866 rows
Reloaded model gives identical 2023 scores: True
Threshold saved: 0.244 | diagnosis codes in dropdown: 410


**Findings.** The chosen forest saves to a 10.5 MB file and, reloaded, reproduces the 2023 scores exactly, so the app will show the same results as this notebook. Every file stays well under GitHub's 100 MB limit, and the config records the threshold (0.244), the cost inputs and their sources' values, and the scikit-learn version the model must be loaded with.

## AGENT LOG
| Step | Agent did | I did | Verified |
|---|---|---|---|
| 1 | N/A | Created repo folder, renamed files into data/, ran cell | Both files found: True / True |
| 2 | Wrote load code | Ran cell, compared output | Shapes (134,596 × 33) and (137,867 × 33) matched against expected output|
| 3 | Wrote data-check code; corrected its own earlier "120 +" count (225 was post-exclusion; raw is 286) | Ran cell, compared output against expected output| All nine checks matched |
| 4 | Wrote cleaning, exclusion and target code | Decided to keep duplicate rows (no patient ID) and to build cutoffs from 2022 only ;Ran cell, compared output | Row counts and base rates matched the checkpoint proposal |
| 5 | Wrote feature/split code; chose min_frequency=30 to match the 30-case rule, and handle_unknown so unseen 2023 codes don't crash | Ran cell, compared output | Shapes, leakage check and unseen-category counts matched |
| 6 | Wrote the scoring function regarding both baselines | Chose the 2022 base rate as the rule's cut-off; Ran cell, compared output | Dummy 79.5% accuracy / 0 caught; rule AUC 0.571 matched |
| 7 | Wrote the logistic regression pipeline, 5-fold CV on 2022, train-vs-test AUC check | Ran cell, compared output | CV mean 0.628, test AUC 0.628 matched |
| 8 | Wrote forest pipeline and CV grid; widened the grid to include 3 after 10 came out at the edge of the first grid | Ran cell, compared output | CV picked 10; test AUC 0.639 matched |
| 9 | Wrote the cost inputs (with sources) and the threshold trade-off on 2022 out-of-fold predictions | Chose capacity-limited at 30% of admissions with the random forest, and the reasoning for it | Output matched; choice and reasoning are mine |
| 10 | Wrote threshold-locking and one-pass 2023 evaluation for all three at 30% capacity | Ran cell, compared output | Threshold 0.244; forest 10,843 caught, $17.15M; +811 stays and +$1.46M vs rule matched |
| 11 | Wrote leakage refit (DRG, severity, risk of mortality) and duplicate-drop refit | Discovered why the leakage gain is small (target relative to DRG x severity); Ran cell, compared output | AUC 0.639 → 0.657 with leaky columns; duplicates move AUC by 0.0004 |
| 12 | Wrote permutation importance (20,000-row test sample, 5 repeats) and the direction tables | Chose permutation over built-in importance; Ran cell, compared output | Facility, age group, admission type as top three matched |
| 13 | Wrote the segment scoring (1,000-stay minimum) | Picked the weakest segment for the model card; Ran cell, compared output | 70 or Older AUC 0.600 vs 0.639 overall matched |
| 14 | Wrote the save step (model, config, scored test rows) and the reload check | Ran cell, confirmed the three new files appear in the repo | Reload gave identical scores; threshold 0.244 saved |
| 14 (rev) | Added the diagnosis-rule score to test_scores.csv and removed rounding, after the app showed 10,782 caught instead of 10,843 | Edited the line, reran all | App now reproduces the notebook's 10,843 / $17.15M exactly |